# Обучение EfficientDet и DETR на Fashionpedia

EfficientDet и DETR **не входят** в `torchvision.models.detection` — у каждой свой forward, свой формат таргета и своя конвенция индексации классов (без фонового +1, в отличие от Faster R-CNN/SSD). Обучение и оценка реализованы отдельными функциями `run_efficientdet_training`/`run_detr_training` в `src/training/train.py`, но метрики считаются той же `evaluate_coco_detector`, что и у остальных 3 моделей (через тонкие инференс-адаптеры).

Порядок: проверка окружения -> клон репо -> данные -> smoke-test (2 эпохи, подвыборка) -> полное обучение EfficientDet -> проверка бюджета времени -> smoke-test и полное обучение DETR -> итоговая таблица из 5 моделей -> сохранение + Save Version.

## 0. Проверка окружения и установка зависимостей

In [ ]:
import sys, subprocess
import torch

print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
        print(f'{pkg}: OK')
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
        print(f'{pkg}: установлен')

_ensure('pycocotools')
_ensure('effdet')
_ensure('timm')

# transformers иногда тянет опциональный torchaudio, который на некоторых
# образах ломается на импорте (несовместимая версия .pyd/.so); torchaudio
# не нужен для DETR/детекции, поэтому при таком сбое просто убираем его.
_ensure('transformers')
try:
    from transformers import DetrForObjectDetection, DetrImageProcessor
    print('transformers DETR import: OK')
except OSError as error:
    print('Сбой импорта transformers (вероятно, конфликт torchaudio):', error)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchaudio'], check=False)
    from transformers import DetrForObjectDetection, DetrImageProcessor
    print('transformers DETR import после удаления torchaudio: OK')


## 1. Клонирование репозитория

При повторных запусках в той же сессии `git clone` падает («destination path already exists») — поэтому: если папка есть, делаем `git pull`, иначе клонируем.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/artemdev2281/cv-fashion-object-detection.git'
REPO_DIR = '/kaggle/working/cv-fashion-object-detection'

if os.path.exists(REPO_DIR):
    print('Репозиторий уже существует в рабочей папке (git pull)')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('sys.path[0]:', sys.path[0])


## 2. Данные (COCO-разметка) и классы

Те же данные, что и для YOLOv8/Faster R-CNN/SSD, подключены через Add Input. COCO-аннотации `coco/{split}.json` + изображения `images/{split}/`. `DATA_ROOT` — явная переменная, `file_name` в JSON — относительный путь, объединяется с `DATA_ROOT` внутри `CocoDetectionDataset` (без изменений).

In [ ]:
import json
from pathlib import Path

DATA_ROOT = Path(
    '/kaggle/input/notebooks/neuromindgpt/'
    'cv-fashion-object-detection/cv-fashion-object-detection/data/processed'
)
assert DATA_ROOT.exists(), f'Не найден корень данных: {DATA_ROOT}. Проверьте Add Input.'

ANN = {s: DATA_ROOT / 'coco' / f'{s}.json' for s in ('train', 'val', 'test')}
for s, p in ANN.items():
    print(f"{s:5} ann: {p}  [{'OK' if p.exists() else 'не найден'}]")

classes = json.load(open(DATA_ROOT / 'classes.json', encoding='utf-8'))
CLASS_NAMES = [classes['names'][str(i)] for i in range(classes['num_classes'])]
NUM_CLASSES = classes['num_classes']  # 11 — БЕЗ фонового +1 (нет фона ни у effdet, ни у DETR)
print('Классов:', NUM_CLASSES, CLASS_NAMES)


## 3. Конфигурация

In [ ]:
from src.utils.utils import load_config

config = load_config(Path(REPO_DIR) / 'configs' / 'default.yaml')
print('seed:', config['training']['seed'])


## 4. EfficientDet: обязательный smoke-test (2 эпохи, подвыборка 300)

Проверяем, что датасет-адаптер (letterbox-ресайз, bbox в `yxyx`, `img_scale`/`img_size`), forward/backward и инференс-адаптер работают и что **loss уменьшается**, прежде чем запускать полный 20-эпочный прогон на 3000 изображениях. `image_size=512` — стандарт для `tf_efficientdet_d0`, `batch_size=4`.

In [ ]:
from src.training.train import run_efficientdet_training

_ = run_efficientdet_training(
    NUM_CLASSES,
    ANN['train'], ANN['test'], DATA_ROOT, config,
    class_names=CLASS_NAMES,
    batch_size=4, epochs=2, image_size=512,
    subset_size=300, eval_subset_size=300,
    project='/kaggle/working/results/logs_smoke',
)
print('EfficientDet smoke-test пройден. Проверьте в логе выше, что loss по эпохам убывает.')


## 5. EfficientDet: полное обучение (20 эпох, подвыборка train ~3000)

Оценка — на **полном test** (4437), как у остальных 4 моделей.

In [ ]:
import time

_t0 = time.time()
effdet_metrics = run_efficientdet_training(
    NUM_CLASSES,
    ANN['train'], ANN['test'], DATA_ROOT, config,
    class_names=CLASS_NAMES,
    batch_size=4, epochs=20, image_size=512,
    subset_size=3000, eval_subset_size=None,
    project='/kaggle/working/results/logs',
)
effdet_minutes = (time.time() - _t0) / 60
print(f'EfficientDet: обучение заняло {effdet_minutes:.1f} мин')
print('test mAP@0.5:', round(effdet_metrics['map50'], 4),
      '| mAP@0.5:0.95:', round(effdet_metrics['map50_95'], 4))


## 6. DETR: проверка бюджета времени перед полным прогоном

In [ ]:
from src.training.train import run_detr_training

_t0 = time.time()
_ = run_detr_training(
    NUM_CLASSES,
    ANN['train'], ANN['test'], DATA_ROOT, config,
    class_names=CLASS_NAMES,
    batch_size=2, epochs=2,
    subset_size=300, eval_subset_size=300,
    project='/kaggle/working/results/logs_smoke',
)
smoke_minutes_per_epoch = (time.time() - _t0) / 60 / 2
print(f'DETR smoke-test пройден: ~{smoke_minutes_per_epoch:.2f} мин/эпоха на 300 изображениях.')
print('Оценка полного прогона (3000 изобр., 20 эпох, грубо x10 по данным x20 по эпохам):',
      f'~{smoke_minutes_per_epoch * 10 * 20 / 60:.1f} ч. Сверьте с остатком GPU-квоты Kaggle.')


## 7. DETR: полное обучение

**По умолчанию `epochs=20`** (как у всех). Если по оценке в предыдущей ячейке бюджет не позволяет — уменьшите `epochs` ЗДЕСЬ (только для DETR).

In [ ]:
DETR_EPOCHS = 20  

_t0 = time.time()
detr_metrics = run_detr_training(
    NUM_CLASSES,
    ANN['train'], ANN['test'], DATA_ROOT, config,
    class_names=CLASS_NAMES,
    batch_size=2, epochs=DETR_EPOCHS,
    subset_size=3000, eval_subset_size=None,
    project='/kaggle/working/results/logs',
)
detr_minutes = (time.time() - _t0) / 60
print(f'DETR: обучение заняло {detr_minutes:.1f} мин ({DETR_EPOCHS} эпох)')
print('test mAP@0.5:', round(detr_metrics['map50'], 4),
      '| mAP@0.5:0.95:', round(detr_metrics['map50_95'], 4))


## 8. Итоговая таблица (5 моделей)

In [ ]:
import pandas as pd

def show(name, m):
    print(f"=== {name} (test) ===")
    print(f"mAP@0.5={m['map50']:.4f}  mAP@0.5:0.95={m['map50_95']:.4f}  "
          f"P={m['precision']:.4f}  R={m['recall']:.4f}  F1={m['f1']:.4f}")
    df = pd.DataFrame(m['per_class']).T[['map50', 'map50_95', 'precision', 'recall', 'f1']]
    return df.sort_values('map50_95', ascending=False).round(4)

df_effdet = show('EfficientDet', effdet_metrics)
df_detr = show('DETR', detr_metrics)
df_effdet


In [ ]:
df_detr

In [ ]:
# Итоговое сравнение всех 5 моделей (точность vs скорость, two-stage vs one-stage vs transformer)
compare = pd.DataFrame({
    'YOLOv8n': {'map50': 0.534, 'map50_95': 0.390, 'precision': 0.581, 'recall': 0.537, 'f1': 0.558},
    'Faster R-CNN': {'map50': 0.565, 'map50_95': 0.384, 'precision': 0.636, 'recall': 0.730, 'f1': 0.680},
    'SSD': {'map50': 0.413, 'map50_95': 0.243, 'precision': 0.640, 'recall': 0.549, 'f1': 0.591},
    'EfficientDet': {k: effdet_metrics[k] for k in ('map50', 'map50_95', 'precision', 'recall', 'f1')},
    'DETR': {k: detr_metrics[k] for k in ('map50', 'map50_95', 'precision', 'recall', 'f1')},
}).T.round(4)
compare


## 9. Сохранение результатов + Save Version

In [ ]:
import json, shutil
from pathlib import Path

out = Path('/kaggle/working/results')
out.mkdir(parents=True, exist_ok=True)

json.dump({'efficientdet': effdet_metrics, 'detr': detr_metrics},
          open(out / 'metrics_effdet_detr.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=2)
df_effdet.to_csv(out / 'efficientdet_per_class.csv', encoding='utf-8')
df_detr.to_csv(out / 'detr_per_class.csv', encoding='utf-8')
compare.to_csv(out / 'final_comparison_5_models.csv', encoding='utf-8')

shutil.make_archive('/kaggle/working/effdet_detr_results', 'zip', '/kaggle/working/results')

for model_name in ('efficientdet', 'detr'):
    d = Path('/kaggle/working/results/logs') / model_name
    pth = list(d.glob('*.pth'))
    print(f"{model_name}: веса {[str(p) for p in pth]} | csv {d / 'results.csv'}")
print()
print('Готово. Скачайте /kaggle/working/effdet_detr_results.zip и НЕ ЗАБУДЬТЕ Save Version.')
